<a href="https://colab.research.google.com/github/Sarasii22/Broadband-Fraud-Management-System/blob/main/Rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/Intern/dataset.parquet")

In [3]:
df.head()

,subscriber_id,record_opening_time,record_closing_time,cc_total_octets_bytes,cc_input_octets_bytes,cc_output_octets_bytes,load_date
0,SUB_365EECB8,2026-05-16 11:00:00,2026-05-16 10:00:00,0,8094,0,2026-05-17 05:04:50
1,SUB_18F7320C,2026-05-16 11:00:00,2026-05-16 10:00:00,0,104908831,3101465,2026-05-17 05:04:50
2,SUB_291441E6,2026-05-16 11:00:00,2026-05-16 10:00:00,0,1265702,0,2026-05-17 05:04:50
3,SUB_DF3164B0,2026-05-16 11:00:00,2026-05-16 10:00:00,0,11306,0,2026-05-17 05:04:50
4,SUB_A7EA0B37,2026-05-16 11:00:00,2026-05-16 10:00:00,0,104970480,576504,2026-05-17 05:04:50


In [6]:
df = df.drop_duplicates()

Bytes to MB

In [7]:
df["download_mb"] = df["cc_input_octets_bytes"]/(1024*1024)
df["upload_mb"] = df["cc_output_octets_bytes"]/(1024*1024)

df["total_usage_mb"] = df["download_mb"] + df["upload_mb"]

df.head()

/tmp/ipykernel_810/864016973.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["download_mb"] = df["cc_input_octets_bytes"]/(1024*1024)
/tmp/ipykernel_810/864016973.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["upload_mb"] = df["cc_output_octets_bytes"]/(1024*1024)
/tmp/ipykernel_810/864016973.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://p

,subscriber_id,record_opening_time,record_closing_time,cc_total_octets_bytes,cc_input_octets_bytes,cc_output_octets_bytes,load_date,download_mb,upload_mb,total_usage_mb
0,SUB_365EECB8,2026-05-16 11:00:00,2026-05-16 10:00:00,0,8094,0,2026-05-17 05:04:50,0.007719,0.000000,0.007719
1,SUB_18F7320C,2026-05-16 11:00:00,2026-05-16 10:00:00,0,104908831,3101465,2026-05-17 05:04:50,100.048858,2.957788,103.006645
2,SUB_291441E6,2026-05-16 11:00:00,2026-05-16 10:00:00,0,1265702,0,2026-05-17 05:04:50,1.207067,0.000000,1.207067
3,SUB_DF3164B0,2026-05-16 11:00:00,2026-05-16 10:00:00,0,11306,0,2026-05-17 05:04:50,0.010782,0.000000,0.010782
4,SUB_A7EA0B37,2026-05-16 11:00:00,2026-05-16 10:00:00,0,104970480,576504,2026-05-17 05:04:50,100.107651,0.549797,100.657448


In [8]:
df.shape

(2681373, 10)

In [9]:
daily_usage = (
    df.groupby(["subscriber_id", "load_date"])["total_usage_mb"]
    .sum()
    .reset_index()
)

In [10]:
daily_usage

,subscriber_id,load_date,total_usage_mb
0,SUB_00000326,2026-05-17 05:04:50,65.675679
1,SUB_00001E72,2026-05-17 05:04:50,4.904803
2,SUB_000048C2,2026-05-17 05:04:50,732.268138
3,SUB_0000573D,2026-05-17 05:04:50,0.260768
4,SUB_00005906,2026-05-17 05:04:50,65.612861
...,...,...,...
512700,SUB_FFFFA359,2026-05-17 05:04:50,4.734103
512701,SUB_FFFFD0A0,2026-05-17 05:04:50,108.456056
512702,SUB_FFFFD502,2026-05-17 05:04:50,1.535686
512703,SUB_FFFFE75F,2026-05-17 05:04:50,335.053882


In [11]:
sessions = (
    df.groupby("subscriber_id")
    .size()
)

In [12]:
sessions

,0
subscriber_id,
SUB_00000326,6
SUB_00001E72,2
SUB_000048C2,17
SUB_0000573D,5
SUB_00005906,3
...,...
SUB_FFFFA359,2
SUB_FFFFD0A0,3
SUB_FFFFD502,6


In [14]:
subscriber_profile = df.groupby("subscriber_id").agg(

    total_download_mb = ("download_mb","sum"),
    total_upload_mb = ("upload_mb","sum"),
    total_usage_mb = ("total_usage_mb","sum"),
    avg_usage_mb = ("total_usage_mb", 'mean'),
    max_usage_mb = ("total_usage_mb", "max"),
    Number_of_sessions = ("record_opening_time","count")


)

Upload Ratio - upload_ratio = total_upload_mb / (total_download_mb + 1)
(+1 avodis division by zero)

In [15]:
subscriber_profile["upload_ratio"] = (
    subscriber_profile["total_upload_mb"] /
    (subscriber_profile["total_download_mb"] + 1)
)

In [16]:
subscriber_profile

,total_download_mb,total_upload_mb,total_usage_mb,avg_usage_mb,max_usage_mb,Number_of_sessions,upload_ratio
subscriber_id,,,,,,,
SUB_00000326,11.273734,54.401945,65.675679,10.945947,50.008046,6,4.432387
SUB_00001E72,4.904803,0.000000,4.904803,2.452402,4.835382,2,0.000000
SUB_000048C2,31.777060,700.491078,732.268138,43.074596,100.134709,17,21.371383
SUB_0000573D,0.193535,0.067233,0.260768,0.052154,0.065182,5,0.056331
SUB_00005906,26.344103,39.268758,65.612861,21.870954,51.285352,3,1.436096
...,...,...,...,...,...,...,...
SUB_FFFFA359,3.766790,0.967313,4.734103,2.367052,4.702164,2,0.202927
SUB_FFFFD0A0,5.640141,102.815915,108.456056,36.152019,102.815915,3,15.483997
SUB_FFFFD502,0.530098,1.005588,1.535686,0.255948,0.365241,6,0.657205


Features for IQR

In [17]:
rule_features = [
    "total_usage_mb",
    "total_download_mb",
    "total_upload_mb",
    "avg_usage_mb",
    "max_usage_mb",
    "Number_of_sessions",
    "upload_ratio"
]

Calculate IQR

In [18]:
iqr_limits = {}

for feature in rule_features:

    Q1 = subscriber_profile[feature].quantile(0.25)
    Q3 = subscriber_profile[feature].quantile(0.75)

    IQR = Q3 - Q1

    upper_limit = Q3 + (1.5 * IQR)

    iqr_limits[feature] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Upper Limit": upper_limit
    }

Thresholds

In [19]:
import pandas as pd

iqr_table = pd.DataFrame(iqr_limits).T

iqr_table

,Q1,Q3,IQR,Upper Limit
total_usage_mb,3.465756,232.323649,228.857893,575.610489
total_download_mb,0.573442,151.562714,150.989271,378.046620
total_upload_mb,0.664155,63.971254,63.307099,158.931903
avg_usage_mb,1.194622,51.821631,50.627009,127.762145
max_usage_mb,2.464637,102.556546,100.091909,252.694410
Number_of_sessions,2.000000,7.000000,5.000000,14.500000
upload_ratio,0.032590,2.077521,2.044931,5.144918


Rule 01

In [20]:
subscriber_profile["Rule_01"] = (
    subscriber_profile["total_usage_mb"] >
    iqr_limits["total_usage_mb"]["Upper Limit"]
).astype(int)

Rule 02

In [21]:
subscriber_profile["Rule_02"] = (
    subscriber_profile["total_download_mb"] >
    iqr_limits["total_download_mb"]["Upper Limit"]
).astype(int)

Rule 03

In [22]:
subscriber_profile["Rule_03"] = (
    subscriber_profile["total_upload_mb"] >
    iqr_limits["total_upload_mb"]["Upper Limit"]
).astype(int)

Rule 04

In [23]:
subscriber_profile["Rule_04"] = (
    subscriber_profile["avg_usage_mb"] >
    iqr_limits["avg_usage_mb"]["Upper Limit"]
).astype(int)

Rule 05

In [24]:
subscriber_profile["Rule_05"] = (
    subscriber_profile["max_usage_mb"] >
    iqr_limits["max_usage_mb"]["Upper Limit"]
).astype(int)

Rule 06

In [25]:
subscriber_profile["Rule_06"] = (
    subscriber_profile["Number_of_sessions"] >
    iqr_limits["Number_of_sessions"]["Upper Limit"]
).astype(int)

Rule 07

In [26]:
subscriber_profile["Rule_07"] = (
    subscriber_profile["upload_ratio"] >
    iqr_limits["upload_ratio"]["Upper Limit"]
).astype(int)

Count Triggered Rules

In [27]:
rule_columns = [
    "Rule_01",
    "Rule_02",
    "Rule_03",
    "Rule_04",
    "Rule_05",
    "Rule_06",
    "Rule_07"
]

subscriber_profile["rule_count"] = subscriber_profile[
    rule_columns
].sum(axis=1)